In [30]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow

DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 3

mlflow.set_tracking_uri("http://localhost:5100")
MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

In [31]:
def pull_test_data():
    x_test_binary = requests.get("http://localhost:8555/test-data/test_images.npy")
    x_test_binary.raise_for_status()
    y_test_binary = requests.get("http://localhost:8555/test-data/test_labels.npy")
    y_test_binary.raise_for_status()
    with open("../../02_data/mnist/test_images.npy", "wb") as f:
        f.write(x_test_binary.content)
    with open("../../02_data/mnist/test_labels.npy", "wb") as f:
        f.write(y_test_binary.content)
    x_test = np.load("../../02_data/mnist/test_images.npy", allow_pickle=True)
    y_test = np.load("../../02_data/mnist/test_labels.npy", allow_pickle=True)
    return x_test, y_test

x_test, y_test = pull_test_data()

np.savez(
    "../../02_data/agg_test/course_test.npz",
    x_test=x_test,
    y_test=y_test,
)

In [ ]:
!docker run --name aggregator_test -d --rm \
    -p 5100:5000 -p 9100:8000 \
    -v "$(realpath ../../02_data/agg_test):/data" \
    ghcr.io/bencefarkas27/aggregator:test-auth

In [33]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response of aggregator: 200 - 


In [34]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "admin"
with open("../../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["admin_password"]
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787749634327, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787749634327, lifecycle_stage='active', name='edge-fl-experiment', tags={}, trace_location=None, workspace='default'>

In [35]:
def create_dummy_model(input_shape):
    model = keras.Sequential([
        keras.layers.InputLayer(shape=input_shape),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    return model

def mlflow_load_model(model_name):
    response = requests.get(f"http://localhost:8555/{model_name}.weights.h5")
    response.raise_for_status()
    weights_path = f"{model_name}.weights.h5"
    with open(weights_path, "wb") as f:
        f.write(response.content)
    input_shape = os.environ.get("MLFLOW_INPUT_SHAPE", (28, 28))
    model = create_dummy_model(input_shape=input_shape)
    model.load_weights(weights_path)
    model.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])
    with mlflow.start_run(run_name=f"{model_name}_run") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(model, f"{model_name}_model")
        return run.info.run_id

def get_run_ids():
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    runs_df = mlflow.search_runs([experiment.experiment_id])
    run_ids = runs_df["run_id"].tolist()
    run_ids.reverse()
    return run_ids


In [41]:
def call_do_agg_with_deltarep():
    aggregation_config = {
        "local_model_hosts": [
            "http://localhost:5000"
            for _ in range(NUM_CLIENTS)
        ],
        "local_model_uris": [
            f"runs:/{run_ids[client_id]}/client_{client_id}_model"
            for client_id in range(NUM_CLIENTS)
        ],
        "local_data_size": [
            len(np.load(f"../../02_data/mnist/client_data/client_{client_id}_train_images.npy"))
            for client_id in range(NUM_CLIENTS)
        ],
        "aggregation_algorithm": "deltarep",
        "global_model_host": "http://localhost:5000",
        "global_model_uri": f"runs:/{run_ids[-1]}/aggregated_model"            
    }
    response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)
    response.raise_for_status()
    return response
    

In [37]:
for client_id in range(NUM_CLIENTS):
        mlflow_load_model(f"client_{client_id}")
mlflow_load_model("aggregated")
run_ids = get_run_ids()
for client_id in range(NUM_CLIENTS):
        x_train_binary = requests.get(f"http://localhost:8555/client_data/client_{client_id}_train_images.npy")
        with open(f"../../02_data/mnist/client_data/client_{client_id}_train_images.npy", "wb") as f:
            f.write(x_train_binary.content)

2026/08/26 15:08:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/26 15:08:55 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run at: http://localhost:5100/#/experiments/1/runs/14790bb04a934910bfe1da63f51d6cf8
🧪 View experiment at: http://localhost:5100/#/experiments/1


2026/08/26 15:09:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/26 15:09:04 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_1_run at: http://localhost:5100/#/experiments/1/runs/9366e019df0c4f13920dffe1ae4e15f8
🧪 View experiment at: http://localhost:5100/#/experiments/1


2026/08/26 15:09:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/26 15:09:13 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_2_run at: http://localhost:5100/#/experiments/1/runs/4d936c139ff4488f8b0cb14c2a26675d
🧪 View experiment at: http://localhost:5100/#/experiments/1


2026/08/26 15:09:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/26 15:09:20 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run aggregated_run at: http://localhost:5100/#/experiments/1/runs/92da447d4fb8458da3ac4ffeaa7c5289
🧪 View experiment at: http://localhost:5100/#/experiments/1


In [42]:
response = call_do_agg_with_deltarep()
print(response.json())

{'aggregated_model_host_external': 'http://localhost:5100', 'aggregated_model_host_organizational': 'http://aggregator:5000', 'aggregated_model_uri': 'models:/m-a05130a2152b442db546b383b8d353c4', 'aggregated_model_username': 'model_user', 'aggregated_model_password': 'hiPMMENYuzajkO_vuq9owKnV3ORxRSapm7GgpNw9BAU'}


In [43]:
# Pull the aggregated model from mlflow
try:
    # Search the experiment and retrieve the runs
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    models = mlflow.search_logged_models(
        experiment_ids=[experiment.experiment_id],
        order_by=[{"field_name": "creation_time", "ascending": False}],
        output_format="list"
    )
    new_aggregate = models[0]
    previous_aggregate = models[1] if len(models) > 1 else None
    new_aggregate_model = mlflow.tensorflow.load_model(f"models:/{new_aggregate.model_id}")
    new_aggregate_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    previous_aggregate_model = mlflow.tensorflow.load_model(f"models:/{previous_aggregate.model_id}")
    previous_aggregate_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    new_aggregated_loss, new_aggregated_accuracy = new_aggregate_model.evaluate(x_test, y_test, verbose=0)
    print(f"New Aggregated Model - Loss: {new_aggregated_loss}, Accuracy: {new_aggregated_accuracy}")

    previous_aggregated_loss, previous_aggregated_accuracy = previous_aggregate_model.evaluate(x_test, y_test, verbose=0)
    print(f"Previous Aggregated Model - Loss: {previous_aggregated_loss}, Accuracy: {previous_aggregated_accuracy}")
except Exception as e:
    print(f"Failed to load model: {e}")


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_21', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


New Aggregated Model - Loss: nan, Accuracy: 0.09799999743700027
Previous Aggregated Model - Loss: 37.268497467041016, Accuracy: 0.2921000123023987


In [29]:
!docker stop aggregator_test

aggregator_test
